# Experiments Analysis

Retrieve and filter LangSmith evaluation runs for flexible post-hoc analysis.

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from typing import Any, Callable, Dict, List, Optional

from langsmith import Client

client = Client()

In [2]:
def load_experiment_runs(
    experiment_prefix: str = "",
    dataset_name: str = "Dataset v1.",
) -> List[Dict[str, Any]]:
    """
    Retrieve all evaluation runs from LangSmith experiments and return them
    as a flat list of dicts ready for filtering and inspection.

    Each record contains:
        experiment         — experiment (project) name
        task_id            — the eval task identifier
        thread_id          — the pipeline checkpoint thread id
        run_id             — LangSmith run id
        scores             — dict of {score_key: float} from all evaluators
        <score_key>        — every score also flattened to top level for easy filtering
        final_code         — generated code from the pipeline state
        retrieval_context  — RAG context from the pipeline state
        outputs            — raw LangSmith run outputs (full pre_computed_state inside)

    Args:
        experiment_prefix: Only include experiments whose name starts with this string.
                           Empty string means include all experiments for the dataset.
        dataset_name:      LangSmith dataset name used to scope experiments.
    """
    records: List[Dict[str, Any]] = []

    projects = list(client.list_projects(reference_dataset_name=dataset_name))
    if experiment_prefix:
        projects = [p for p in projects if p.name.startswith(experiment_prefix)]

    print(f"Found {len(projects)} experiment(s) matching prefix '{experiment_prefix}'")

    for project in projects:
        runs = list(client.list_runs(project_name=project.name, is_root=True))

        if not runs:
            continue

        # Bulk-fetch all feedback for this project in one call
        run_ids = [str(r.id) for r in runs]
        feedback_by_run: Dict[str, Dict[str, float]] = {rid: {} for rid in run_ids}
        for fb in client.list_feedback(run_ids=run_ids):
            rid = str(fb.run_id)
            if fb.score is not None:
                feedback_by_run[rid][fb.key] = fb.score

        for run in runs:
            rid = str(run.id)
            scores = feedback_by_run.get(rid, {})
            outputs = run.outputs or {}
            pre_state = outputs.get("pre_computed_state") or {}

            task_id = (
                str(outputs.get("task_id") or "").strip()
                or str(pre_state.get("task_id") or "").strip()
            )

            record: Dict[str, Any] = {
                "experiment": project.name,
                "task_id": task_id,
                "thread_id": outputs.get("thread_id", ""),
                "run_id": rid,
                "scores": scores,
                "final_code": pre_state.get("final_code") or pre_state.get("generated_code") or "",
                "retrieval_context": pre_state.get("retrieval_context", ""),
                "outputs": outputs,
            }
            # Flatten scores to top level for convenient filtering
            record.update(scores)

            records.append(record)

    print(f"Loaded {len(records)} total run(s)")
    return records

In [3]:
def filter_runs(
    records: List[Dict[str, Any]],
    **conditions: Any,
) -> List[Dict[str, Any]]:
    """
    Filter records by field values.

    Each keyword argument is a field name. The value can be:
    - A scalar     → exact equality match  (e.g., code_execution_score=0.0)
    - A callable   → predicate on the field (e.g., task_id=lambda v: "add" in v)

    Missing fields evaluate to None against the condition.

    Example:
        filter_runs(records, code_statements_score=1.0, code_execution_score=0.0)
        filter_runs(records, task_id=lambda v: v.startswith("add"))
    """
    result = []
    for record in records:
        if all(
            (cond(record.get(key)) if callable(cond) else record.get(key) == cond)
            for key, cond in conditions.items()
        ):
            result.append(record)
    return result


def display_runs(
    records: List[Dict[str, Any]],
    show_code: bool = False,
    show_context: bool = False,
    code_preview_chars: int = 600,
    context_preview_chars: int = 400,
) -> None:
    """
    Pretty-print a list of run records.

    Args:
        records:              Output of load_experiment_runs / filter_runs.
        show_code:            Print a preview of final_code.
        show_context:         Print a preview of retrieval_context.
        code_preview_chars:   Max chars to show for code preview.
        context_preview_chars: Max chars to show for context preview.
    """
    SEP = "─" * 70
    print(f"{len(records)} record(s)\n{SEP}")

    for i, r in enumerate(records, 1):
        scores = r.get("scores", {})
        score_str = "  ".join(
            f"{k}={v:.2f}" for k, v in sorted(scores.items()) if v is not None
        )
        print(f"\n[{i}]  task_id={r.get('task_id', '?')!r}")
        print(f"     experiment: {r.get('experiment', '?')}")
        print(f"     thread_id:  {r.get('thread_id', '?')}")
        print(f"     scores:     {score_str or '(none)'}")

        if show_code:
            code = r.get("final_code", "") or ""
            snippet = code[:code_preview_chars]
            ellipsis = "..." if len(code) > code_preview_chars else ""
            print(f"     final_code:\n{snippet}{ellipsis}")

        if show_context:
            ctx = r.get("retrieval_context", "") or ""
            snippet = ctx[:context_preview_chars]
            ellipsis = "..." if len(ctx) > context_preview_chars else ""
            print(f"     retrieval_context:\n{snippet}{ellipsis}")

    print(f"\n{SEP}")

## Usage Examples

In [4]:
# --- Load ---
# Empty prefix loads all experiments for the dataset.
# Supply a prefix string to scope to a specific experiment family.
EXPERIMENT_PREFIX = "2) With concision + summary"       # e.g. "2) With concision"
DATASET_NAME = "Dataset v1."

records = load_experiment_runs(
    experiment_prefix=EXPERIMENT_PREFIX,
    dataset_name=DATASET_NAME,
)

Found 1 experiment(s) matching prefix '2) With concision + summary'
Loaded 6 total run(s)


In [8]:
# --- Filter: code statements correct but code didn't execute ---
interesting = filter_runs(
    records,
    code_statements_score=1.0,
    code_execution_score=0.0,
)

In [12]:
display_runs(interesting, show_code=True, show_context=True, code_preview_chars=20000, context_preview_chars=20000)

3 record(s)
──────────────────────────────────────────────────────────────────────

[1]  task_id='add_toggle_to_page'
     experiment: 2) With concision + summary (1000 tokens) (New Baseline).-fa58e382
     thread_id:  add_toggle_to_page_20260310123540
     scores:     code_execution_score=0.00  code_statements_score=1.00  rag_statements_score=0.67
     final_code:
import os
import dotenv
import requests
import sys

dotenv.load_dotenv()

def add_toggle(page_id: str, title: str, content: str, notion_version: str = "2022-06-28", integration_token: str = os.getenv("NOTION_TOKEN")) -> dict:
    """
    Adds a toggle block with nested content to a specified Notion page.
    """
    url = f"https://api.notion.com/v1/pages/{page_id}/children"
    headers = {
        "Authorization": f"Bearer {integration_token}",
        "Content-Type": "application/json",
        "Notion-Version": notion_version
    }
    
    payload = {
        "children": [
            {
                "object": "block",

In [ ]:
# --- Other filter patterns ---

# All runs where RAG retrieval was perfect
# filter_runs(records, rag_statements_score=1.0)

# Runs for a specific task using a predicate
# filter_runs(records, task_id=lambda v: "add_task" in str(v))

# Runs where code execution passed
# filter_runs(records, code_execution_score=1.0)

# Any field from the pipeline state is accessible via record["outputs"]["pre_computed_state"]
# Example: look at retrieval_context for a specific run
# r = records[0]
# print(r["retrieval_context"])
# print(r["final_code"])
# print(r["outputs"])  # full raw LangSmith output dict